In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geobr
import geopandas as gpd
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Lasso
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import shap


# --- CONFIGURAÇÃO DE ESTILO (PRETO E BRANCO) ---
%matplotlib inline
plt.rcParams.update({
    'axes.titlesize': 0,
    'axes.labelsize': 12,
    'grid.alpha': 0.3,
    'figure.facecolor': 'white'
})

# 1. CARGA DOS DADOS
df = pd.read_csv('base_enriquecida_tcc_v2.csv')
df = df.drop(columns=['id_equipe_data', 'equipe_data_id', 'datahora_inicio','datahora_fim'], errors='ignore')
alvo = 'duracao_real_tempo_servico'


# 2. SANEAMENTO E AMOSTRAGEM
df_analise = df.dropna(subset=[alvo]).copy()
if len(df_analise) > 100000:
    df_analise = df_analise.sample(100000, random_state=42)

# 3. FILTRAGEM AUTOMÁTICA DE ATRIBUTOS
# Filtra colunas que inviabilizam o cálculo (IDs e chaves únicas)
atributos_candidatos = []
for col in df_analise.columns:
    if col == alvo:
        continue
    
    # Regra: Se a coluna tem mais de 90% de valores únicos, ela é um ID ou 
    # variável contínua de alta precisão que quebra a lógica discreta do MI.
    if df_analise[col].nunique() > (len(df_analise) * 0.9):
        continue
        
    atributos_candidatos.append(col)

# 4. TRANSFORMAÇÃO
print(f"Normalizando {len(atributos_candidatos)} atributos...")
for col in atributos_candidatos:
    df_analise[col] = LabelEncoder().fit_transform(df_analise[col].astype(str))

# 5. EXECUÇÃO DO TORNEIO GLOBAL
X = df_analise[atributos_candidatos].values
y = df_analise[alvo].values

print(f">>> Calculando Relevância (MI) para {len(atributos_candidatos)} atributos...")
# discrete_features=True trata as variáveis como categorias, acelerando o processo
scores = mutual_info_regression(X, y, discrete_features=True, random_state=42)

# 6. CONSOLIDAÇÃO DO RANKING
df_ranking = pd.DataFrame({'Atributo': atributos_candidatos, 'Score_MI': scores})
df_ranking = df_ranking.sort_values(by='Score_MI', ascending=False)

print("\n" + "="*50)
print("RANKING: AS VARIÁVEIS MAIS PODEROSAS DA OPERAÇÃO")
print("="*50)
print(df_ranking.head(10))

# 1. FILTRANDO APENAS COLUNAS NUMÉRICAS PARA O TESTE RÁPIDO
# (Inclua aqui qualquer coluna suspeita que você queira auditar)
df_pearson = df.select_dtypes(include=[np.number])

# 2. CÁLCULO DA CORRELAÇÃO COM O ALVO
# Vamos focar apenas no que se correlaciona com a duração real
correlacoes = df_pearson.corr(method='pearson')[['duracao_real_tempo_servico']].sort_values(
    by='duracao_real_tempo_servico', ascending=False
)

print("\n" + "="*80)
print("AUDITORIA INICIAL: DETECÇÃO DE LEAKAGE VIA PEARSON")
print("="*80)

# 3. EXIBIÇÃO FORMATADA
# Variáveis com corr > 0.9 ou < -0.9 são fortes candidatas a serem removidas
display(correlacoes.style.background_gradient(cmap='Greys'))

In [ ]:
# 1. IDENTIFICA O TOP 5 (Garante que o mapeamento seja consistente)
top_5_servicos = df['tipo_de_servico'].value_counts().head(5).index.tolist()

# 2. CRIA O MAPEAMENTO
mapeamento_s = {servico: f'S{i+1}' for i, servico in enumerate(top_5_servicos)}

# 3. CRIA A COLUNA QUE ESTÁ FALTANDO
df['servico_anon'] = df['tipo_de_servico'].map(mapeamento_s)

# Opcional: verifique se funcionou
print("Mapeamento realizado:", mapeamento_s)
print(df['servico_anon'].value_counts())

In [ ]:
# 1. IDENTIFICAÇÃO DO RANKING E ANONIMIZAÇÃO DINÂMICA
# Ordenamos todos os serviços por volume para definir S1, S2... até S10
ranking_full = df['tipo_de_servico'].value_counts().index.tolist()

# Criamos o dicionário: o mais frequente vira S1, o segundo S2, etc.
mapping_dinamico = {servico: f'S{i+1}' for i, servico in enumerate(ranking_full) if i < 10}

# Aplicamos a anonimização. O que não for Top 10 vira "S-Outros"
df['servico_anon'] = df['tipo_de_servico'].map(mapping_dinamico).fillna('S-Outros')

# 2. PREPARAÇÃO DOS DADOS PARA O GRÁFICO
pareto_data = df['servico_anon'].value_counts().reset_index()
pareto_data.columns = ['Serviço', 'Frequência']

# Garantimos a ordem S1 -> S10 -> S-Outros no eixo X
ordem_eixo = [f'S{i}' for i in range(1, 11)] + ['S-Outros']
pareto_data['Serviço'] = pd.Categorical(pareto_data['Serviço'], categories=ordem_eixo, ordered=True)
pareto_data = pareto_data.sort_values('Serviço')

# Cálculo do Percentual Acumulado
pareto_data['Acumulado'] = (pareto_data['Frequência'].cumsum() / pareto_data['Frequência'].sum()) * 100

# 3. PLOTAGEM (PRETO SÓLIDO, SEM TÍTULO DO GRÁFICO)
fig, ax1 = plt.subplots(figsize=(12, 6))

# Barras Pretas (Volume)
ax1.bar(pareto_data['Serviço'].head(10), pareto_data['Frequência'].head(10),color='#333333', edgecolor='black')
ax1.set_ylabel('Volume de Ocorrências')
ax1.set_xlabel('Tipos de Serviço (Categorias Anonimizadas)') # Rótulo do eixo X adicionado

# Eixo Secundário para a Linha de Pareto
ax2 = ax1.twinx()
ax2.plot(pareto_data['Serviço'].head(10), pareto_data['Acumulado'].head(10), color='black', marker='o', linewidth=1.5)
ax2.axhline(80, color='black', linestyle='--', linewidth=1) # Linha de 80%

ax2.set_ylabel('Percentual Acumulado (%)')
ax2.set_ylim(0, 110)

# Remoção de grids pesados (limpeza visual para o LaTeX)
ax1.grid(False)
ax2.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# 1. FILTRAGEM PARA OS 5 SERVIÇOS MAIS REPRESENTATIVOS
top_5 = [f'S{i}' for i in range(1, 6)]
df_boxplot = df[df['servico_anon'].isin(top_5)].copy()

# 2. CONFIGURAÇÃO DA FIGURA
plt.figure(figsize=(10, 6))

# 3. PLOTAGEM DO BOXPLOT (ESCALA LINEAR)
sns.boxplot(
    x='servico_anon', 
    y='duracao_real_tempo_servico', 
    data=df_boxplot, 
    order=top_5,
    color='white',             # Fundo da caixa branco para impressão
    linewidth=1.5,             # Contorno preto sólido
    fliersize=2,               # Outliers visíveis na escala real
    medianprops=dict(color="black", linewidth=2)
)

# 4. RÓTULOS E GRADE
plt.ylabel('Duração Real do Serviço (min)')
plt.xlabel('Tipos de Serviço (Top 5 Anonimizados)')

# Grade leve apenas no eixo Y para facilitar a leitura dos tempos
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:


# 1. Carregar a malha dos estados (simplificada para performance)
# Isso desenha o contorno do Brasil/Estados ao fundo
mapa_base = geobr.read_state(simplified=True)

# 2. Definir o escopo dos serviços (Top 5 identificados no Pareto)
top_5 = [f'S{i}' for i in range(1, 6)]
df_focado = df[df['servico_anon'].isin(top_5)].copy()

# 3. Configuração do estilo acadêmico
plt.style.use('seaborn-v0_8-paper') 
fig, ax = plt.subplots(figsize=(10, 8))

# 4. Mapa ao fundo (Branco com contorno preto fino)
mapa_base.plot(ax=ax, facecolor='white', edgecolor='black', linewidth=0.4)

# 5. Scatter plot com escala de cinzas e marcadores variados
# Diferenciamos por tom de cinza (hue) e formato do ponto (style)
sns.scatterplot(
    data=df_focado,
    x='longitude', 
    y='latitude',
    hue='servico_anon', 
    hue_order=top_5,
    style='servico_anon', 
    palette=['black', '#404040', '#808080', '#A0A0A0', '#C0C0C0'],
    alpha=0.4, 
    s=10,       # Tamanho levemente maior para destacar os símbolos (triângulo, bola, x)
    ax=ax
)

# 6. Limites Geográficos e Proporção
# Ajustado para o território brasileiro conforme seu snippet
ax.set_xlim([-74, -34])
ax.set_ylim([-34, 6])
ax.set_aspect('equal') 

# 7. Rótulos dos Eixos (Sem título na figura para LaTeX)
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)

# Ajuste da Legenda Acadêmica (Posicionada fora do mapa)
plt.legend(
    title='Categorias de Serviço', 
    bbox_to_anchor=(1.02, 1), 
    loc='upper left', 
    frameon=True
)

plt.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()

# 8. Salvar para o projeto
# plt.savefig('dispersao_geografica_S1_S5.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:


# ==============================================================================
# 4.1. ETAPA DE PRIVACIDADE: ANONIMIZAÇÃO DE NOMES
# ==============================================================================
print(">>> Anonimizando colunas de identificação pessoal...")

for col in ['nome', 'nome_completo']:
    if col in df.columns:
        # Criamos um ranking de nomes para que o anonimato seja consistente
        nomes_unicos = df[col].unique()
        mapeamento_nomes = {nome: f"Filial/Empresa {i+1}" for i, nome in enumerate(nomes_unicos)}
        df[col] = df[col].map(mapeamento_nomes)

# ==============================================================================
# 4.2. FUNÇÕES DE CARACTERIZAÇÃO (S1 A S5)
# ==============================================================================

def extrair_tendencia_perfil(serie):
    """
    Realiza a extração de tendência central para dados quantitativos 
    ou frequência relativa (Top 10) para dados qualitativos.
    """
    if pd.api.types.is_numeric_dtype(serie):
        if serie.empty: return "0.00"
        return f"Média: {serie.mean():.2f}"
    
    # Extração das 10 categorias mais frequentes (Top 10)
    frequencia = serie.value_counts(normalize=True).head(10)
    if frequencia.empty: return "-"
    
    # Formatação com quebra de linha para melhor leitura na matriz
    return "\n".join([f"{cat} ({val*100:.1f}%)" for cat, val in frequencia.items()])

def gerar_matriz_percentil(df_origem, percentil_alvo, superior=True):
    """Gera matriz comparativa de atributos por tipologia de serviço (S1 a S5)."""
    
    # Escopo restrito solicitado
    SERVICOS_TARGET = ['S1', 'S2', 'S3', 'S4', 'S5']
    
    # Selecionamos TODAS as variáveis disponíveis, removendo apenas as de apoio e o alvo
    VARIAVEIS_DIAGNOSTICO = [col for col in df_origem.columns if col not in ['duracao_real_tempo_servico', 'servico_anon']]
    
    matriz = pd.DataFrame(index=VARIAVEIS_DIAGNOSTICO, columns=SERVICOS_TARGET)
    
    for s in SERVICOS_TARGET:
        df_s = df_origem[df_origem['servico_anon'] == s]
        if df_s.empty: continue
        
        limite = df_s['duracao_real_tempo_servico'].quantile(percentil_alvo)
        
        # Filtro de cauda: Superior (P95) ou Inferior (P05)
        if superior:
            df_filtro = df_s[df_s['duracao_real_tempo_servico'] >= limite]
        else:
            df_filtro = df_s[df_s['duracao_real_tempo_servico'] <= limite]
        
        for var in VARIAVEIS_DIAGNOSTICO:
            if var in df_filtro.columns:
                matriz.loc[var, s] = extrair_tendencia_perfil(df_filtro[var])
                
    return matriz

# ==============================================================================
# 4.3. EXECUÇÃO E ESTILIZAÇÃO
# ==============================================================================

print(">>> Gerando Matrizes de Caracterização Completa...")

matriz_p05 = gerar_matriz_percentil(df, 0.05, superior=False)
matriz_p95 = gerar_matriz_percentil(df, 0.95, superior=True)

estilo_matriz = {
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'border': '1px solid black',
    'font-size': '11px',
    'vertical-align': 'top',
    'background-color': 'white',
    'color': 'black'
}

print("\n" + "="*80)
print("MATRIZ DE CARACTERIZAÇÃO: PERCENTIL 05 (EFICIÊNCIA SUPERIOR)")
print("="*80)
display(matriz_p05.style.set_properties(**estilo_matriz))

print("\n" + "="*80)
print("MATRIZ DE CARACTERIZAÇÃO: PERCENTIL 95 (LATÊNCIA CRÍTICA)")
print("="*80)
display(matriz_p95.style.set_properties(**estilo_matriz))

In [ ]:

# 1. CONFIGURAÇÕES DE ESCALA E ESCOPO
FATOR_ESCALA = 0.05  # Ajuste conforme a densidade da sua base
SERVICOS_MODELO = ['S1', 'S2', 'S3', 'S4', 'S5']

def plotar_mapa_bolhas_tcc(df_input, nome_arquivo, titulo_legenda, is_p95=True):
    # --- Filtragem por Percentil Dinâmico (S1 a S5) ---
    lista_percentil = []
    for s in SERVICOS_MODELO:
        df_s = df_input[df_input['servico_anon'] == s]
        if df_s.empty: continue
            
        q = 0.95 if is_p95 else 0.05
        limite = df_s['duracao_real_tempo_servico'].quantile(q)
        
        if is_p95:
            df_sel = df_s[df_s['duracao_real_tempo_servico'] >= limite]
        else:
            df_sel = df_s[df_s['duracao_real_tempo_servico'] <= limite]
        lista_percentil.append(df_sel)
    
    df_final = pd.concat(lista_percentil).copy()

    # --- Agregação Espacial (Binning para evitar sobreposição) ---
    # Agrupa em blocos de coordenadas para criar as bolhas de frequência
    df_final['lat_bin'] = df_final['latitude'].apply(lambda x: round(x * 2) / 2)
    df_final['lon_bin'] = df_final['longitude'].apply(lambda x: round(x * 2) / 2)
    df_ag = df_final.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='frequencia')

    # --- Plotagem Acadêmica (Preto e Branco) ---
    # Carrega todos os estados de forma simplificada (evita o erro de sigla)
    estados = geobr.read_state(simplified=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    estados.plot(ax=ax, facecolor='#f9f9f9', edgecolor='black', linewidth=0.5)

    scatter = ax.scatter(
        df_ag['lon_bin'], df_ag['lat_bin'],
        s=df_ag['frequencia'] * FATOR_ESCALA, 
        color='black', alpha=0.6, edgecolor='white', linewidth=0.3
    )

    # --- Tratamento da Legenda de Tamanho ---
    handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, num=4)
    labels_limpos = []
    for l in labels:
        try:
            # Extração do valor numérico para reverter a escala do fator
            val = float(l.replace('$\\mathdefault{', '').replace('}$', '').replace('{', '').replace('}', ''))
            labels_limpos.append(str(int(val / FATOR_ESCALA)))
        except:
            labels_limpos.append(l)

    ax.legend(handles, labels_limpos, title=titulo_legenda, loc="lower left", 
              bbox_to_anchor=(0.05, 0.1), frameon=True, fontsize='small', title_fontsize='small')

    # Limites geográficos do Brasil (ajustar se sua operação for regional)
    ax.set_xlim([-74, -34])
    ax.set_ylim([-34, 6])
    ax.set_aspect('equal')
    
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    
    # Salva para uso no LaTeX
    #plt.savefig(f'{nome_arquivo}.png', dpi=300, bbox_inches='tight')
    plt.show()

# 2. GERAÇÃO DOS MAPAS PARA O TCC
print(">>> Gerando mapas de bolhas para auditoria espacial...")
plotar_mapa_bolhas_tcc(df, 'mapa_bolhas_p95', 'Qtd. Casos P95 (Atraso)', is_p95=True)
plotar_mapa_bolhas_tcc(df, 'mapa_bolhas_p05', 'Qtd. Casos P05 (Eficiência)', is_p95=False)

In [ ]:


# 1. Definições de Escopo
SERVICOS_TARGET = ['S1', 'S2', 'S3', 'S4', 'S5']
alvo = 'duracao_real_tempo_servico'

# Selecionamos as variáveis disponíveis (IDs já foram removidos no início do notebook)
# Removemos apenas o alvo e a coluna de apoio para o cálculo
colunas_ignorar = [alvo, 'servico_anon', 'id_servico', 'id_proc', 'id_equipe', 'resource_id']
VARIAVEIS_DIAGNOSTICO = [col for col in df.columns if col not in colunas_ignorar]

matriz_mi = pd.DataFrame(index=VARIAVEIS_DIAGNOSTICO, columns=SERVICOS_TARGET)

print(">>> Calculando MI Score por serviço (S1 a S5)...")

for s in SERVICOS_TARGET:
    df_s = df[df['servico_anon'] == s].copy()
    
    # Amostragem para agilidade
    if len(df_s) > 50000:
        df_s = df_s.sample(50000, random_state=42)
    
    X_s = df_s[VARIAVEIS_DIAGNOSTICO].copy()
    y_s = df_s[alvo]
    
    # Identificação de discretas e encoding
    discrete_mask = []
    for col in VARIAVEIS_DIAGNOSTICO:
        if not pd.api.types.is_numeric_dtype(X_s[col]):
            X_s[col] = LabelEncoder().fit_transform(X_s[col].astype(str))
            discrete_mask.append(True)
        else:
            X_s[col] = X_s[col].fillna(0)
            discrete_mask.append(True if "int" in str(X_s[col].dtype) else False)

    mi_scores = mutual_info_regression(X_s, y_s, discrete_features=discrete_mask, random_state=42)
    matriz_mi[s] = mi_scores

# --- PROCESSAMENTO PARA O HEATMAP ---

# 1. Converter para float e preencher nulos
matriz_plot = matriz_mi.astype(float).fillna(0)

# 2. FILTRO SOLICITADO: Manter apenas variáveis com MI > 0.01 em pelo menos um serviço
matriz_plot = matriz_plot[matriz_plot.max(axis=1) > 0.01]

# 3. Normalização por coluna (Serviço) para o gradiente de cinza
matriz_norm = (matriz_plot - matriz_plot.min()) / (matriz_plot.max() - matriz_plot.min())

# 4. Plotagem (Estilo TCC)
plt.figure(figsize=(12, 10))
sns.heatmap(matriz_norm, 
            annot=matriz_plot, # Exibe o valor real do MI
            fmt=".3f", 
            cmap="Greys", 
            linewidths=.5, 
            linecolor='lightgrey',
            cbar=False)

plt.xlabel('Serviços (Categorias Anonimizadas)')
plt.ylabel('Atributos Operacionais e Geográficos')
plt.tight_layout()
plt.show()




In [ ]:

# 1. PREPARAÇÃO
# Usamos o mesmo dataframe filtrado (S1 a S5) e sem as variáveis de vazamento
dict_lasso_final = {}

print(">>> Calculando coeficientes Lasso para identificação de relevância linear...")

for s in SERVICOS_TARGET:
    df_s = df[df['servico_anon'] == s].copy()
    
    # Amostragem para estabilidade
    if len(df_s) > 50000:
        df_s = df_s.sample(50000, random_state=42)
    
    X_s = df_s[VARIAVEIS_DIAGNOSTICO].copy()
    y_s = df_s[alvo]
    
    # Encoding para o Lasso (precisa ser numérico)
    for col in VARIAVEIS_DIAGNOSTICO:
        if not pd.api.types.is_numeric_dtype(X_s[col]):
            X_s[col] = LabelEncoder().fit_transform(X_s[col].astype(str))
        X_s[col] = X_s[col].fillna(0)
    
    # O Lasso exige escalonamento (StandardScaler) para comparar coeficientes de grandezas diferentes
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_s)
    
    # Alpha 0.1 para forçar a seleção (penalização L1)
    modelo_lasso = Lasso(alpha=0.1, random_state=42)
    modelo_lasso.fit(X_scaled, y_s)
    
    # Extração dos coeficientes absolutos
    importancia = pd.Series(np.abs(modelo_lasso.coef_), index=VARIAVEIS_DIAGNOSTICO)
    
    # Selecionamos apenas as Top 5 que não zeraram
    dict_lasso_final[s] = importancia.sort_values(ascending=False).head(10).index.tolist()

# 2. EXIBIÇÃO DA MATRIZ LASSO
df_lasso_visual = pd.DataFrame(dict_lasso_final)
df_lasso_visual.index = [f'{i+1}ª Variável' for i in range(10)]

print("\n" + "="*80)
print("TOP 5 VARIÁVEIS CRÍTICAS: SELEÇÃO VIA LASSO (POR SERVIÇO)")
print("="*80)
display(df_lasso_visual.style.set_properties(**estilo_matriz))

In [ ]:
# 1. PREPARAÇÃO DAS FEATURES (Filtro MI > 0.01)
# Usamos a matriz_plot que geramos no bloco anterior
features_finais = matriz_plot.index.tolist()
alvo = 'duracao_real_tempo_servico'

# 2. FUNÇÃO AUXILIAR COM TRATAMENTO DE CATEGORIAS DESCONHECIDAS
def treinar_e_avaliar(df_treino, df_teste, features):
    X_train = df_treino[features].copy()
    y_train = df_treino[alvo]
    X_test = df_teste[features].copy()
    y_test = df_teste[alvo]

    # Identifica colunas categóricas
    cat_features = [col for col in features if not pd.api.types.is_numeric_dtype(X_train[col])]
    
    if cat_features:
        # OrdinalEncoder com tratamento de valores não vistos no treino
        # Ele atribui -1 para qualquer categoria nova que aparecer no teste
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        
        X_train[cat_features] = encoder.fit_transform(X_train[cat_features].astype(str))
        X_test[cat_features] = encoder.transform(X_test[cat_features].astype(str))
    
    # Preenchimento de nulos numéricos remanescentes
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    # Modelo XGBoost
    modelo = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
    modelo.fit(X_train, y_train)
    
    preds = modelo.predict(X_test)
    return mean_absolute_error(y_test, preds)

# 3. EXECUÇÃO DA COMPARAÇÃO
resultados_mae = []

print(">>>Modelos (Global vs Específicos)...")

# Base consolidada S1 a S5
df_all = df[df['servico_anon'].isin(['S1', 'S2', 'S3', 'S4', 'S5'])].copy()

# --- MODELO GLOBAL ---
train_g, test_g = train_test_split(df_all, test_size=0.2, random_state=42)
# O MAE Global total é apenas para referência
mae_global_total = treinar_e_avaliar(train_g, test_g, features_finais)

# --- MODELOS ESPECÍFICOS ---
for s in ['S1', 'S2', 'S3', 'S4', 'S5']:
    df_s = df_all[df_all['servico_anon'] == s]
    train_s, test_s = train_test_split(df_s, test_size=0.2, random_state=42)
    
    # MAE do modelo treinado só para este serviço
    mae_especifico = treinar_e_avaliar(train_s, test_s, features_finais)
    
    # MAE do modelo global aplicado especificamente neste serviço (comparação real)
    # Avaliamos o conjunto de teste deste serviço usando o conhecimento do treino global
    mae_global_no_servico = treinar_e_avaliar(train_g, test_s, features_finais)
    
    resultados_mae.append({
        'Serviço': s,
        'MAE Modelo Global': mae_global_no_servico,
        'MAE Modelo Específico': mae_especifico,
        'Melhoria (%)': ((mae_global_no_servico - mae_especifico) / mae_global_no_servico) * 100
    })

# 4. EXIBIÇÃO DOS RESULTADOS
df_comp = pd.DataFrame(resultados_mae)
print("\n" + "="*80)
print("RESULTADO FINAL: MODELAGEM SEGMENTADA vs GLOBAL")
print("="*80)
display(df_comp.round(2))

# 5. GRÁFICO DE BARRAS ACADÊMICO
plt.figure(figsize=(10, 6))
x = np.arange(len(df_comp['Serviço']))
width = 0.35

plt.bar(x - width/2, df_comp['MAE Modelo Global'], width, label='Global', color='lightgrey', edgecolor='black')
plt.bar(x + width/2, df_comp['MAE Modelo Específico'], width, label='Específico', color='black')

plt.ylabel('MAE (minutos)')
plt.xlabel('Serviços')
plt.xticks(x, df_comp['Serviço'])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:


# 1. DEFINIÇÃO DOS COMPETIDORES E VARIÁVEIS
modelos_torneio = {
    "Lasso (Baseline)": Lasso(alpha=0.1, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
}

# Certifique-se de que servico_anon existe (mapeamento feito anteriormente)
servicos = ['S1', 'S2', 'S3', 'S4', 'S5']
resultados_bench = []

print(">>> Iniciando Torneio de Algoritmos por Serviço (Especialistas)...")

for s in servicos:
    print(f"Avaliando {s}...")
    df_s = df_all[df_all['servico_anon'] == s].copy()
    
    # Split individual por serviço
    X_s = df_s[features_finais].copy()
    y_s = df_s[alvo]
    X_train, X_test, y_train, y_test = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
    
    # Encoding para cada serviço
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    cat_cols = [c for c in features_finais if not pd.api.types.is_numeric_dtype(X_train[c])]
    if cat_cols:
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols].astype(str))
        X_test[cat_cols] = enc.transform(X_test[cat_cols].astype(str))
    
    X_train, X_test = X_train.fillna(0), X_test.fillna(0)

    for nome, model in modelos_torneio.items():
        t_inicio = time.time()
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)
        
        resultados_bench.append({
            "Serviço": s,
            "Algoritmo": nome,
            "MAE (min)": round(mae, 3),
            "RMSE (min)": round(rmse, 3),
            "R²": round(r2, 4),
            "Tempo (s)": round(time.time() - t_inicio, 2)
        })

# 2. QUADRO COMPARATIVO COMPLETO
df_bench_total = pd.DataFrame(resultados_bench)
print("\n" + "="*80)
print("QUADRO GERAL DE PERFORMANCE POR SERVIÇO")
print("="*80)
display(df_bench_total.sort_values(by=["Serviço", "MAE (min)"]))

In [ ]:



# 1. Configuração dos modelos para o teste de sensibilidade
# Modelo focado em MAE (reg:absoluteerror)
model_mae = xgb.XGBRegressor(
    objective='reg:absoluteerror', 
    tree_method='hist',
    n_estimators=1000,
    learning_rate=0.05
)

# Modelo focado em RMSE (reg:squarederror - Padrão)
model_rmse = xgb.XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    n_estimators=1000,
    learning_rate=0.05
)

# 2. Treinamento
model_mae.fit(X_train, y_train)
model_rmse.fit(X_train, y_train)

# 3. Predições
preds_mae = model_mae.predict(X_test)
preds_rmse = model_rmse.predict(X_test)

# 4. Cálculo das métricas estatísticas
mae_final_mae = mean_absolute_error(y_test, preds_mae)
rmse_final_mae = np.sqrt(mean_squared_error(y_test, preds_mae))

mae_final_rmse = mean_absolute_error(y_test, preds_rmse)
rmse_final_rmse = np.sqrt(mean_squared_error(y_test, preds_rmse))

# 5. Lógica de "Eventos Justificados" (Ganho Operacional)
# Consideramos um evento justificado quando o resíduo contextual é 
# significativamente menor que o desvio da média histórica (auditoria tradicional)
limite_auditoria = 15 # Exemplo: auditoria flagava tudo acima de 15min de desvio
residuos_mae = np.abs(y_test - preds_mae)
residuos_rmse = np.abs(y_test - preds_rmse)

eventos_just_mae = np.sum(residuos_mae < limite_auditoria)
eventos_just_rmse = np.sum(residuos_rmse < limite_auditoria)

# 6. Consolidação dos Resultados para a Tabela
tabela_sensibilidade = pd.DataFrame({
    'Métrica de Treinamento': ['MAE (reg:absoluteerror)', 'RMSE (reg:squarederror)'],
    'MAE Final': [mae_final_mae, mae_final_rmse],
    'RMSE Final': [rmse_final_mae, rmse_final_rmse],
    'Eventos Justificados': [eventos_just_mae, eventos_just_rmse],
    'Ganho Operacional (%)': [(eventos_just_mae/len(y_test))*100, (eventos_just_rmse/len(y_test))*100]
})

print(tabela_sensibilidade)

In [ ]:

# 1. ESPAÇO DE BUSCA
param_dist = {
    'n_estimators': [200, 400, 600],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.7, 0.8]
}

resultados_tuning = []
modelos_otimizados = {} # Dicionário para guardar os modelos finais de cada serviço

print(">>> Iniciando Tuning dos Hiperparâmetros para os 5 serviços...")

for s in servicos:
    print(f"Otimizando modelo para {s}...")
    
    # Preparação dos dados (repetindo o split para garantir isolamento)
    df_s = df_all[df_all['servico_anon'] == s].copy()
    X_s = df_s[features_finais].copy()
    y_s = df_s[alvo]
    X_train, X_test, y_train, y_test = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
    
    # Encoding
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    if cat_cols:
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols].astype(str))
        X_test[cat_cols] = enc.transform(X_test[cat_cols].astype(str))
    
    X_train, X_test = X_train.fillna(0), X_test.fillna(0)

    # Execução do Tuning
    rs = RandomizedSearchCV(XGBRegressor(random_state=42, n_jobs=-1), 
                            param_distributions=param_dist, 
                            n_iter=15, scoring='neg_mean_absolute_error', 
                            cv=3, random_state=42, n_jobs=-1)
    
    rs.fit(X_train, y_train)
    
    # Avaliação do melhor modelo encontrado
    modelo_final = rs.best_estimator_
    modelos_otimizados[s] = modelo_final # Salva o modelo para uso posterior
    
    preds_final = modelo_final.predict(X_test)
    
    mae_f = mean_absolute_error(y_test, preds_final)
    rmse_f = np.sqrt(mean_squared_error(y_test, preds_final))
    r2_f = r2_score(y_test, preds_final)
    
    resultados_tuning.append({
        "Serviço": s,
        "Melhores Parâmetros": rs.best_params_,
        "MAE Final": round(mae_f, 3),
        "RMSE Final": round(rmse_f, 3),
        "R² Final": round(r2_f, 4)
    })

# 2. RESULTADO CONSOLIDADO DA OTIMIZAÇÃO
df_tuning_final = pd.DataFrame(resultados_tuning)
print("\n" + "="*80)
print("RESULTADO DA OTIMIZAÇÃO FINAL (S1-S5)")
print("="*80)
display(df_tuning_final)

In [ ]:
# 1. COMPARAÇÃO DE GANHO FINAL
resultados_finais_bench = []

for s in servicos:
    # Recupera o Lasso (do bloco de torneio) e o XGBoost Tunado
    # Assumindo que você tem os resultados do torneio guardados em df_bench_total
    mae_lasso = df_bench_total[(df_bench_total['Serviço'] == s) & (df_bench_total['Algoritmo'] == 'Lasso (Baseline)')]['MAE (min)'].values[0]
    mae_ia = df_tuning_final[df_tuning_final['Serviço'] == s]['MAE Final'].values[0]
    
    resultados_finais_bench.append({
        'Serviço': s,
        'MAE Lasso (Base)': mae_lasso,
        'MAE IA Otimizada': mae_ia,
        'Ganho sobre Base (%)': ((mae_lasso - mae_ia) / mae_lasso) * 100
    })

df_ganho_global = pd.DataFrame(resultados_finais_bench)
print(">>> GANHO GLOBAL: IA ESPECIALISTA VS BASELINE LINEAR")
display(df_ganho_global.round(2))

In [ ]:


importancia_por_servico = []

print(">>> Gerando Matriz SHAP para todos os serviços...")

for s in servicos:
    modelo = modelos_otimizados[s]
    # Amostra de teste específica do serviço
    df_s = df_all[df_all['servico_anon'] == s].copy()
    _, te_s = train_test_split(df_s, test_size=0.2, random_state=42)
    X_te = te_s[features_finais].copy()
    
    # Encoder
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    cat_cols = [c for c in features_finais if not pd.api.types.is_numeric_dtype(X_te[c])]
    X_te[cat_cols] = enc.fit_transform(X_te[cat_cols].astype(str))
    X_te = X_te.fillna(0)
    
    explainer = shap.TreeExplainer(modelo)
    shap_vals = explainer.shap_values(X_te)
    
    # Média absoluta por feature
    mean_shap = np.abs(shap_vals).mean(axis=0)
    importancia_por_servico.append(mean_shap)

# Criar DataFrame da Matriz
df_shap_matrix = pd.DataFrame(importancia_por_servico, columns=features_finais, index=servicos).T

print("\n>>> HEATMAP DE IMPORTÂNCIA (SHAP VALUES POR SERVIÇO)")
display(df_shap_matrix.style.background_gradient(cmap='Greys', axis=0).format("{:.4f}"))

In [ ]:
# 1. CONSOLIDAÇÃO DE PREDIÇÕES E RESIDUAIS
df_analise_erro = pd.DataFrame()

print(">>> Calculando residuais para os modelos especialistas (S1-S5)...")

for s in ['S1', 'S2', 'S3', 'S4', 'S5']:
    # Usando o nome da sua variável: modelos_otimizados
    modelo = modelos_otimizados[s]
    df_s = df_all[df_all['servico_anon'] == s].copy()
    
    # Split de Teste (Semente 42 para manter a consistência com o treino)
    _, te_s = train_test_split(df_s, test_size=0.2, random_state=42)
    
    # Preparação dos dados de teste
    X_te = te_s[features_finais].copy()
    
    # Encoder (Tratando colunas categóricas)
    cat_cols = [c for c in features_finais if not pd.api.types.is_numeric_dtype(X_te[c])]
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_te[cat_cols] = enc.fit_transform(X_te[cat_cols].astype(str))
    
    X_te = X_te.fillna(0)
    
    # Cálculo: Real - Predito
    te_s['predicao_ia'] = modelo.predict(X_te)
    te_s['residual'] = te_s['duracao_real_tempo_servico'] - te_s['predicao_ia']
    
    # Empilhando para a análise final
    df_analise_erro = pd.concat([df_analise_erro, te_s])

# 2. IDENTIFICAÇÃO DAS FRONTEIRAS DE ERRO (Novas Pontas)
p05_res = df_analise_erro['residual'].quantile(0.05)
p95_res = df_analise_erro['residual'].quantile(0.95)

print("\n" + "="*60)
print("LIMITES DE ERRO DO MODELO IA (RESIDUAIS)")
print("="*60)
print(f"P05 (Superestimados): {p05_res:.2f} min")
print(f"P95 (Subestimados):   {p95_res:.2f} min")
print("="*60)

In [ ]:
# 1. FUNÇÃO DE EXTRAÇÃO (Conforme sua padronização anterior)
def extrair_tendencia_perfil(serie):
    if pd.api.types.is_numeric_dtype(serie):
        if serie.empty: return "0.00"
        return f"Média: {serie.mean():.2f}"
    
    frequencia = serie.value_counts(normalize=True).head(10)
    if frequencia.empty: return "-"
    
    return "\n".join([f"{cat} ({val*100:.1f}%)" for cat, val in frequencia.items()])

# 2. FUNÇÃO GERADORA DA MATRIZ DE RESIDUAIS
def gerar_matriz_diagnostico_ia(df_residuais, percentil_alvo, superior=True):
    """
    Gera matriz comparativa baseada nos RESIDUAIS (Erro da IA).
    """
    SERVICOS_TARGET = ['S1', 'S2', 'S3', 'S4', 'S5']
    # Analisamos apenas as variáveis que entraram no modelo
    VARIAVEIS_ANÁLISE = features_finais 
    
    matriz = pd.DataFrame(index=VARIAVEIS_ANÁLISE, columns=SERVICOS_TARGET)
    
    for s in SERVICOS_TARGET:
        df_s = df_residuais[df_residuais['servico_anon'] == s]
        if df_s.empty: continue
        
        # O corte é feito no residual do serviço específico
        limite_res = df_s['residual'].quantile(percentil_alvo)
        
        if superior:
            # P95: Subestimados (Real > Predito) - IA achou que seria rápido
            df_filtro = df_s[df_s['residual'] >= limite_res]
        else:
            # P05: Superestimados (Real < Predito) - IA achou que demoraria
            df_filtro = df_s[df_s['residual'] <= limite_res]
        
        for var in VARIAVEIS_ANÁLISE:
            if var in df_filtro.columns:
                matriz.loc[var, s] = extrair_tendencia_perfil(df_filtro[var])
                
    return matriz

# 3. GERAÇÃO DAS MATRIZES PARA O TCC
print("\n" + "="*80)
print("MATRIZ DE DIAGNÓSTICO: SUBESTIMADOS (P95 - IA Subestimou a Demora)")
print("="*80)
matriz_p95_res = gerar_matriz_diagnostico_ia(df_analise_erro, 0.95, superior=True)
display(matriz_p95_res.style.set_properties(**{
    'text-align': 'left', 
    'white-space': 'pre-wrap', 
    'border': '1px solid black',
    'font-size': '10px'
}))

print("\n" + "="*80)
print("MATRIZ DE DIAGNÓSTICO: SUPERESTIMADOS (P05 - IA Exagerou na Estimativa)")
print("="*80)
matriz_p05_res = gerar_matriz_diagnostico_ia(df_analise_erro, 0.05, superior=False)
display(matriz_p05_res.style.set_properties(**{
    'text-align': 'left', 
    'white-space': 'pre-wrap', 
    'border': '1px solid black', 
    'font-size': '10px'
}))

In [ ]:


# 1. DEFINIÇÃO DOS RESÍDUOS
# O resíduo é a base da anomalia contextual (Equação 12, pág. 17) [cite: 375, 376]
df_analise_erro['residuo'] = df_analise_erro[alvo] - df_analise_erro['predicao_ia']

# 2. CÁLCULO DOS LIMITES (P95) POR SERVIÇO
# O critério de auditoria usa o percentil 95 (P95) [cite: 516, 519]
print("Calculando thresholds P95 por serviço...")

# Limite para o Sistema Atual (baseado na duração real)
thresholds_sistema = df_analise_erro.groupby('servico_anon')[alvo].quantile(0.95).to_dict()

# Limite para a IA (baseado nos resíduos positivos)
thresholds_ia = df_analise_erro.groupby('servico_anon')['residuo'].quantile(0.95).to_dict()

# 3. CRIAÇÃO DAS COLUNAS DE STATUS (O que estava faltando!)
# Definimos 'Erro' para Anomalias e 'Acerto' para Normalidade para bater com seu dicionário
df_analise_erro['status_sistema'] = df_analise_erro.apply(
    lambda x: 'Erro' if x[alvo] > thresholds_sistema[x['servico_anon']] else 'Acerto', axis=1
)

df_analise_erro['status_ia'] = df_analise_erro.apply(
    lambda x: 'Erro' if x['residuo'] > thresholds_ia[x['servico_anon']] else 'Acerto', axis=1
)

# 4. MAPEAMENTO DE NOMENCLATURA TÉCNICA (Seu código original)
mapeamento_status = {
    ('Acerto', 'Acerto'): 'Normalidade Mantida',
    ('Erro', 'Acerto'): 'Anomalia -> Normal (Casos Justificados)',
    ('Acerto', 'Erro'): 'Normal -> Anomalia (Casos Novos)',
    ('Erro', 'Erro'): 'Anomalia Persistente'
}

df_analise_erro['transicao_anomalia'] = df_analise_erro.apply(
    lambda x: mapeamento_status[(x['status_sistema'], x['status_ia'])], axis=1
)

# 5. GERAÇÃO DA MATRIZ DE TRANSIÇÃO (Tabela 12, pág. 47) 
df_visual = df_analise_erro.copy()
df_visual['Sistema Atual'] = df_visual['status_sistema'].replace({'Acerto': 'Normal', 'Erro': 'Anômalo (P95)'})
df_visual['Modelo IA'] = df_visual['status_ia'].replace({'Acerto': 'Normal', 'Erro': 'Anômalo'})

matriz_anomalias = pd.crosstab(
    df_visual['Sistema Atual'], 
    df_visual['Modelo IA'],
    margins=True,
    margins_name="Total"
)

# 6. TABELA DE COMPOSIÇÃO POR SERVIÇO (Tabela 13, pág. 48) 
# Ordenando as colunas para facilitar a leitura
colunas_ordem = [
    'Anomalia -> Normal (Casos Justificados)', 
    'Anomalia Persistente', 
    'Normal -> Anomalia (Casos Novos)', 
    'Normalidade Mantida'
]

composicao_anomalias = pd.crosstab(
    df_visual['servico_anon'], 
    df_visual['transicao_anomalia'],
    normalize='index'
) * 100

# Reordenar se todas as colunas existirem
colunas_presentes = [c for c in colunas_ordem if c in composicao_anomalias.columns]
composicao_anomalias = composicao_anomalias[colunas_presentes]

print("\n" + "="*80)
print("MATRIZ DE TRANSIÇÃO GLOBAL (ESTÁTICA vs CONTEXTUAL)")
print("="*80)
display(matriz_anomalias)

print("\n" + "="*80)
print("DISTRIBUIÇÃO PERCENTUAL POR SERVIÇO (%)")
print("="*80)
display(composicao_anomalias.round(2).style.background_gradient(cmap='Greys'))

In [ ]:


# 1. FILTRAGEM: Isolar apenas as pontas críticas (Remover a Normalidade Mantida)
# Criamos um DataFrame contendo apenas os casos de interesse da auditoria
df_somente_pontas = df_analise_erro[df_analise_erro['transicao_anomalia'] != 'Normalidade Mantida'].copy()

# 2. TABELA 1: Contagem Absoluta de Casos na Fila de Auditoria
tabela_pontas_absoluta = pd.crosstab(
    df_somente_pontas['servico_anon'],
    df_somente_pontas['transicao_anomalia'],
    margins=True,
    margins_name="Total da Fila"
)

# 3. TABELA 2: Distribuição Percentual INTERNA da Fila de Auditoria
# O normalize='index' aqui vai fazer com que a soma das 3 pontas dê 100% por linha
colunas_ordem_pontas = [
    'Anomalia -> Normal (Casos Justificados)', 
    'Anomalia Persistente', 
    'Normal -> Anomalia (Casos Novos)'
]

tabela_pontas_percentual = pd.crosstab(
    df_somente_pontas['servico_anon'],
    df_somente_pontas['transicao_anomalia'],
    normalize='index'
) * 100

# Reordenar colunas para manter o padrão descritivo
colunas_presentes_pontas = [c for c in colunas_ordem_pontas if c in tabela_pontas_percentual.columns]
tabela_pontas_percentual = tabela_pontas_percentual[colunas_presentes_pontas]

# 4. EXIBIÇÃO DOS RESULTADOS
print("\n" + "="*80)
print("VOLUMES ABSOLUTOS APENAS NA FILA DE AUDITORIA (SEM NORMALIDADE)")
print("="*80)
display(tabela_pontas_absoluta)

print("\n" + "="*80)
print("COMPOSIÇÃO PERCENTUAL INTERNA DA FILA DE AUDITORIA (%)")
print("="*80)
display(tabela_pontas_percentual.round(2).style.background_gradient(cmap='Greys'))

In [ ]:


# 1. CONFIGURAÇÕES DE ESCALA E ESCOPO
FATOR_ESCALA = 0.5  
SERVICOS_MODELO = ['S1', 'S2', 'S3', 'S4', 'S5']

def plotar_mapa_residuos_ia(df_input, titulo_legenda, is_p95=True):
    # --- Ajuste do nome da coluna (residuo em vez de residual) ---
    col_residuo = 'residuo' # Conforme definido no bloco da matriz de transição
    
    # --- Filtragem por Percentil Dinâmico nos RESIDUOS ---
    lista_percentil = []
    for s in SERVICOS_MODELO:
        df_s = df_input[df_input['servico_anon'] == s].copy()
        if df_s.empty: continue
            
        # P95 = IA subestimou (Anomalia Crítica / Ineficiência)
        # P05 = IA superestimou (Eficiência Superior / Erro de registro)
        q = 0.95 if is_p95 else 0.05
        limite = df_s[col_residuo].quantile(q)
        
        if is_p95:
            df_sel = df_s[df_s[col_residuo] >= limite]
        else:
            df_sel = df_s[df_s[col_residuo] <= limite]
        lista_percentil.append(df_sel)
    
    df_final = pd.concat(lista_percentil).copy()

    # --- Agregação Espacial ---
    df_final['lat_bin'] = df_final['latitude'].apply(lambda x: round(x * 2) / 2)
    df_final['lon_bin'] = df_final['longitude'].apply(lambda x: round(x * 2) / 2)
    df_ag = df_final.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='frequencia')

    # --- Plotagem Acadêmica ---
    print(f"Lendo malha do Brasil para o mapa: {'P95' if is_p95 else 'P05'}...")
    estados = geobr.read_state(simplified=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    estados.plot(ax=ax, facecolor='#f9f9f9', edgecolor='black', linewidth=0.5)

    scatter = ax.scatter(
        df_ag['lon_bin'], df_ag['lat_bin'],
        s=df_ag['frequencia'] * FATOR_ESCALA, 
        color='black', alpha=0.6, edgecolor='white', linewidth=0.3
    )

    # --- Tratamento da Legenda ---
    handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, num=4)
    labels_limpos = []
    for l in labels:
        try:
            # Limpeza das tags LaTeX do Matplotlib para exibir números puros
            val_str = l.replace('$\\mathdefault{', '').replace('}$', '').replace('{', '').replace('}', '')
            val = float(val_str)
            labels_limpos.append(str(int(val / FATOR_ESCALA)))
        except:
            labels_limpos.append(l)

    ax.legend(handles, labels_limpos, title=titulo_legenda, loc="lower left", 
              bbox_to_anchor=(0.05, 0.1), frameon=True, fontsize='small', title_fontsize='small')

    ax.set_xlim([-74, -34])
    ax.set_ylim([-34, 6])
    ax.set_aspect('equal')
    
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.show()

# 2. EXECUÇÃO DAS IMAGENS (O QUE ESTAVA FALTANDO)

# Mapa para a Figura 11: Latência Crítica Inexplicada (P95)
plotar_mapa_residuos_ia(df_analise_erro, titulo_legenda="Novas Pontas P95\n(Resíduo Alto)", is_p95=True)

# Mapa para a Figura 12: Eficiência Contextual Superior (P05)
plotar_mapa_residuos_ia(df_analise_erro, titulo_legenda="Novas Pontas P05\n(Resíduo Baixo)", is_p95=False)

In [ ]:
# 1. DEFINIÇÃO DAS TIPOLOGIAS DE TRANSIÇÃO
tipologias = [
    'Anomalia -> Normal (Casos Justificados)', 
    'Anomalia Persistente', 
    'Normal -> Anomalia (Casos Novos)'
]

# 2. FUNÇÃO PARA GERAR MATRIZ DETALHADA POR TRANSIÇÃO
def gerar_matriz_detalhada_transicao(df_erro, categoria_alvo):
    """
    Gera uma matriz de diagnóstico focada em uma categoria específica 
    de transição entre Sistema Atual e IA.
    """
    SERVICOS_TARGET = ['S1', 'S2', 'S3', 'S4', 'S5']
    # Focamos nas variáveis que entraram no modelo
    VARIAVEIS_ANÁLISE = features_finais 
    
    matriz = pd.DataFrame(index=VARIAVEIS_ANÁLISE, columns=SERVICOS_TARGET)
    
    for s in SERVICOS_TARGET:
        # Filtro: Serviço + Categoria de Impacto
        df_filtro = df_erro[
            (df_erro['servico_anon'] == s) & 
            (df_erro['transicao_anomalia'] == categoria_alvo)
        ]
        
        if df_filtro.empty: 
            continue
        
        for var in VARIAVEIS_ANÁLISE:
            if var in df_filtro.columns:
                matriz.loc[var, s] = extrair_tendencia_perfil(df_filtro[var])
                
    return matriz

# 3. EXECUÇÃO E EXIBIÇÃO
for tipo in tipologias:
    print("\n" + "="*80)
    print(f"DIAGNÓSTICO: {tipo.upper()}")
    print("="*80)
    
    matriz_result = gerar_matriz_detalhada_transicao(df_analise_erro, tipo)
    
    display(matriz_result.style.set_properties(**{
        'text-align': 'left', 
        'white-space': 'pre-wrap', 
        'border': '1px solid black',
        'font-size': '10px'
    }))

In [ ]:
# 1. DEFINIÇÃO DOS SERVIÇOS
servicos = ['S1', 'S2', 'S3', 'S4', 'S5']
dados_confronto_final = []

# 2. CONSOLIDAÇÃO DOS RESULTADOS
for s in servicos:
    # Recupera o MAE do Lasso "completo"
    mae_base = df_bench_total[(df_bench_total['Serviço'] == s) & 
                              (df_bench_total['Algoritmo'].str.contains('Lasso'))]['MAE (min)'].values[0]
    
    # Recupera o MAE do XGBoost Tunado
    mae_ia = df_tuning_final[df_tuning_final['Serviço'] == s]['MAE Final'].values[0]
    
    dados_confronto_final.append({
        'Serviço': s,
        'MAE Base (min)': mae_base,
        'MAE IA Contextual (min)': mae_ia,  # Nome definido aqui
        'Melhoria Real (%)': ((mae_base - mae_ia) / mae_base) * 100
    })

# 3. CRIAÇÃO DO DATAFRAME DA TABELA 14
df_tabela_14 = pd.DataFrame(dados_confronto_final)

# 4. CÁLCULO DA LINHA GLOBAL (Corrigido para acessar a coluna certa)
global_base = df_tabela_14['MAE Base (min)'].mean()
global_ia = df_tabela_14['MAE IA Contextual (min)'].mean()  # <-- O AJUSTE FOI AQUI
global_melhoria = ((global_base - global_ia) / global_base) * 100

df_global = pd.DataFrame([{
    'Serviço': 'Global',
    'MAE Base (min)': global_base,
    'MAE IA Contextual (min)': global_ia,
    'Melhoria Real (%)': global_melhoria
}])

# 5. UNIÃO FINAL
df_tabela_14_final = pd.concat([df_tabela_14, df_global], ignore_index=True)

# EXIBIÇÃO
print(">>> TABELA 14: COMPARATIVO DE PRECISÃO (RECONCILIADA)")
df_tabela_14_final.round(2)

In [ ]:

# Configuração de estilo acadêmico em tons de cinza
estilo_disponivel = plt.style.available
estilo_plot = 'seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in estilo_disponivel else 'default'
plt.style.use(estilo_plot)

plt.figure(figsize=(10, 6))

# Criando o boxplot dos resíduos por serviço
sns.boxplot(
    data=df_analise_erro,
    x='servico_anon',
    y='residuo',
    order=['S1', 'S2', 'S3', 'S4', 'S5'],
    color='lightgray',
    fliersize=2,          # Tamanho dos pontos outliers
    linewidth=1.2,        # Espessura das linhas da caixa
    width=0.5             # Largura da caixa
)

# Adicionando uma linha horizontal de referência no zero (resíduo ideal)
plt.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)

# Configurações de eixos e títulos acadêmicos
plt.title('')
plt.xlabel('Categoria de Serviço Anonimizada')
plt.ylabel('Resíduo Preditivo (minutos)')

plt.tight_layout()
plt.savefig('boxplot_residuos_servicos.png', dpi=300)
plt.show()

In [ ]:
# 1. CONTAGEM TOTAL DA BASE DE TESTE
total_casos = len(df_analise_erro)

# 2. CONTAGEM NA PONTA DO SISTEMA TRADICIONAL (Gráfico B do seu Histograma)
casos_sistema_p95 = (df_analise_erro['status_sistema'] == 'Erro').sum()
pct_sistema = (casos_sistema_p95 / total_casos) * 100

# 3. CONTAGEM NA PONTA DA IA CONTEXTUAL (Gráfico D do seu Histograma)
casos_ia_p95 = (df_analise_erro['status_ia'] == 'Erro').sum()
pct_ia = (casos_ia_p95 / total_casos) * 100

# 4. EXIBIÇÃO DOS RESULTADOS
print("="*60)
print("VERIFICAÇÃO DE VOLUMES ABSOLUTOS NAS PONTAS CRÍTICAS")
print("="*60)
print(f"Total de registros na base de teste: {total_casos:,}")
print(f"Casos no gráfico B (Sistema Tradicional): {casos_sistema_p95:,} ({pct_sistema:.2f}%)")
print(f"Casos no gráfico D (IA Contextual):      {casos_ia_p95:,} ({pct_ia:.2f}%)")
print("="*60)

In [ ]:

print(pd.crosstab(df_visual['Sistema Atual'], df_visual['Modelo IA'], margins=True))